# MN8 Energy Style Portfolio Analytics Project

This notebook simulates the kind of portfolio, project, and operational analytics work described in the MN8 Energy Data Analytics Intern posting.

**Goal:** turn project, engineering, schedule, and financial data into clear portfolio insights for an EPMO-style team.

> Note: the dataset used here is synthetic and built for portfolio demonstration purposes.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

BASE_DIR = Path("..").resolve()
RAW_DIR = BASE_DIR / "data" / "raw"

## 1. Load data

In [ ]:
portfolio = pd.read_csv(RAW_DIR / "project_portfolio.csv", parse_dates=["planned_start", "planned_cod", "forecast_cod"])
monthly = pd.read_csv(RAW_DIR / "monthly_progress.csv", parse_dates=["month"])
issues = pd.read_csv(RAW_DIR / "issue_log.csv", parse_dates=["opened_date", "closed_date"])
portfolio.head()

## 2. Portfolio KPIs

In [ ]:
kpis = {
    "total_projects": len(portfolio),
    "total_capacity_mw": round(portfolio["capacity_mw"].sum(), 1),
    "avg_schedule_variance_days": round(portfolio["schedule_variance_days"].mean(), 1),
    "projects_at_risk": int((portfolio["portfolio_health"] == "At Risk").sum()),
    "avg_budget_variance_pct": round(portfolio["budget_variance_pct"].mean(), 2),
}
pd.DataFrame([kpis])

## 3. Risk and schedule views

In [ ]:
priority_projects = portfolio.sort_values(["risk_score", "schedule_variance_days"], ascending=False)[["project_id", "project_name", "technology", "stage", "risk_score", "schedule_variance_days", "budget_variance_pct"]].head(10)
priority_projects

In [ ]:
capacity = portfolio.groupby("technology")["capacity_mw"].sum().sort_values(ascending=False)
plt.figure(figsize=(8,4.5))
capacity.plot(kind="bar")
plt.title("Portfolio Capacity by Technology")
plt.ylabel("MW")
plt.tight_layout()
plt.show()

In [ ]:
top_delays = portfolio.sort_values("schedule_variance_days", ascending=False).head(10)
plt.figure(figsize=(9,4.5))
plt.bar(top_delays["project_id"], top_delays["schedule_variance_days"])
plt.title("Top 10 Projects by Schedule Delay")
plt.ylabel("Delay (days)")
plt.tight_layout()
plt.show()

## 4. Root-cause analysis

In [ ]:
root_cause_summary = issues.groupby("root_cause", as_index=False).agg(issue_count=("issue_id", "count"), total_impact_days=("impact_days", "sum"), total_impact_cost_usd=("impact_cost_usd", "sum")).sort_values(["total_impact_days", "issue_count"], ascending=False)
root_cause_summary.head(10)

## 5. Recommendations

- Tighten escalation on projects showing both schedule slippage and above-plan spend.
- Track recurring delay drivers such as permitting, equipment lead times, and interconnection.
- Standardize PMO reporting templates so engineering, finance, and operations use the same KPI definitions.
- Review projects with weak progress against plan for recovery actions and governance follow-up.